In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
# 📥 Upload file
from google.colab import files
uploaded = files.upload()

# 📂 Check uploaded files
import os
print("Files in directory:", os.listdir())

# 📊 Load dataset
import pandas as pd

# Automatically pick uploaded file
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# 👀 Preview
print(df.head())
print("Shape:", df.shape)
print(df['sentiment'].value_counts())

Saving IMDB Dataset.csv to IMDB Dataset.csv
Files in directory: ['.config', 'IMDB Dataset.csv', 'sample_data']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Shape: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [13]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [14]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

In [15]:
df['clean_text'] = df['review'].apply(preprocess_text)

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [16]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text']).toarray()

In [17]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text']).toarray()

In [18]:
encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])  # positive=1, negative=0

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [20]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [21]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [23]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [24]:
def evaluate(y_test, y_pred, model_name):
    print(f"\n{model_name} Performance:")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))

In [25]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")


Logistic Regression Performance:
Accuracy : 0.8841
Precision: 0.873795761078998
Recall   : 0.8999801547926176
F1 Score : 0.8866946915632027

Naive Bayes Performance:
Accuracy : 0.8458
Precision: 0.8431795878312071
Recall   : 0.8525501091486406
F1 Score : 0.8478389579632919

Decision Tree Performance:
Accuracy : 0.7117
Precision: 0.7169014084507043
Recall   : 0.7070847390355229
F1 Score : 0.7119592366869817


In [26]:
def predict_sentiment(text):
    processed = preprocess_text(text)
    vector = tfidf.transform([processed]).toarray()
    prediction = lr.predict(vector)

    return "Positive" if prediction[0] == 1 else "Negative"


# Test
print(predict_sentiment("This movie was amazing!"))

Positive


### Insights

- Logistic Regression performed best due to handling high-dimensional sparse data.
- TF-IDF performed better than Bag of Words.
- Preprocessing steps like stopword removal and stemming improved accuracy.
- Decision Tree showed lower performance due to overfitting.
